In [25]:

class Node():
    def __init__(self, type, left, right = None, word=None):
        self.type = type
        self.left = left
        self.right = right
        self.word = word

    def _recursive_repr(self, tab=0):
        s =  f"{'----' * tab}{self.type}"
        if self.word is not None:
            s += f" - {self.word}"
        s+= "\n"
        if self.left is not None:
            s+=  f"{self.left._recursive_repr(tab+1)}"
        if self.right is not None:
            s+=  f"{self.right._recursive_repr(tab+1)}"
        return s

    def __str__(self):
        return self._recursive_repr() + "\n"

    def __repr__(self,):
        return self._recursive_repr() + "\n"

class CYK():
    
    def __init__(self, grammar):
        """ A simple implementation of CYK algorithm
            Parameter:
                grammar : list
                    a list of production rule in chumsky normal form
                    each rule is defined by a tuple (P, (L,R)) with P
                    a non-terminal and L and R respectivelly the right 
                    and left of the production.

            Usage example: 
                sentence = ["The", "cat", "sat",  "on", "the", "couch"]
                pos =      [["DET"], ["NOUN"], ["VERB"], ["PREP"], ["DET"], ["NOUN"]]

                # defining the grammar rules
                G = [
                    # S is axiom
                    ("S", ("NP", "VP")), 
                    
                        # non terminal rules
                    ("NP", ("DET", "NOUN")),
                    ("PP", ("PREP", "NP")),
                    ("VP", ("VERB", "PP")),
                    
                    # terminal (if using pos directly)
                    ("DET", ("DET",)),
                    ("VERB", ("VERB",)),
                    ("NOUN", ("NOUN",)),
                    ("PREP", ("PREP",))
                ]
                cyk = CYK(G)
                cyk(pos, sentence) 

                ## SHOULD RETURN
                [S
                ----NP
                --------DET - The
                --------NOUN - cat
                ----VP
                --------VERB - sat
                --------PP
                ------------PREP - on
                ------------NP
                ----------------DET - the
                ----------------NOUN - couch
                ]
        """
        self.grammar = grammar

    
    def __call__(self, input_list, str):
        """ Apply the CYK algorithm on an input.

            Parameter: 
                input_list: the list of terminal (in our case a list)

            Return:
                List of tree that are recognized by the language
        """
        size = len(input_list)
        cyk_tab = [ [[] for _ in range(i+1)] for i in range(len(input_list)) ][::-1]
        
        # process terminals
        for i, tags_list in enumerate(input_list):
            for tag in tags_list:
                # for all terminal rules
                for production, rule in self.grammar: 
                    if(len(rule) == 1) and (tag in rule):
                        cyk_tab[0][i].append(Node(production, None, word=str[i]))
        for cyk_depth in range(2, size+1):
            left_levels = list(range(1, cyk_depth))
            right_levels = list(range(1, cyk_depth)[::-1])
            for start in range(0, size - cyk_depth + 1):
                for left_level, right_level in zip(left_levels, right_levels):
                    left_start = start
                    right_start = start + left_level
                    Al, Bl = cyk_tab[left_level-1][left_start], cyk_tab[right_level-1][right_start]
                    for p, j in self.grammar:
                        if(len(j) == 2):
                            l, r = j
                            A = [a for a in Al if(a.type == l)]
                            B = [b for b in Bl if(b.type == r)]
                            cyk_tab[cyk_depth-1][start] += [Node(p, a, b ) for a in A  for b in B]

        return [i for i in cyk_tab[-1][0]]


In [26]:
# Grammar in Chomsky Normal Form.
# The two PP rules model the attachment ambiguity of "with".
G = [
    ("ROOT", ("S", "PUNCT")),
    ("S", ("NP", "VP")),
    ("NP", ("DET", "NOUN")),
    ("NP", ("NOUN",)),
    ("NP", ("PROPN", "PROPN")),
    ("NP", ("NP", "PP")),
    ("PP", ("ADP", "NP")),
    ("VP", ("VERB", "NP")),
    ("VP", ("VERB", "PP")),
    ("VP", ("VP", "PP")),
    ("NP", ("NOUN", "NOUN")),
    ("NP", ("NOUN", "PP")),
    ("DET", ("DET",)),
    ("NOUN", ("NOUN",)),
    ("PROPN", ("PROPN",)),
    ("VERB", ("VERB",)),
    ("ADP", ("ADP",)),
    ("PUNCT", ("PUNCT",)),
]

sentences = {
    "The cat sat on the couch.": [
        ["DET"], ["NOUN"], ["VERB"], ["ADP"], ["DET"], ["NOUN"], ["PUNCT"]
    ],
    "Time flies like an arrow.": [
        ["NOUN"], ["VERB"], ["ADP"], ["DET"], ["NOUN"], ["PUNCT"]
    ],
    "The spy saw the cop with the telescope.": [
        ["DET"], ["NOUN"], ["VERB"], ["DET"], ["NOUN"],
        ["ADP"], ["DET"], ["NOUN"], ["PUNCT"]
    ],
    "The spy saw the cop with the revolver.": [
        ["DET"], ["NOUN"], ["VERB"], ["DET"], ["NOUN"],
        ["ADP"], ["DET"], ["NOUN"], ["PUNCT"]
    ],
}

cyk = CYK(G)
for sentence, tags in sentences.items():
    tokens = sentence.replace(".", " .").split()
    trees = cyk(tags, tokens)
    print(f"{sentence} -> {len(trees)} parse(s)")
    for tree in trees:
        print(tree)


The cat sat on the couch. -> 1 parse(s)
ROOT
----S
--------NP
------------DET - The
------------NOUN - cat
--------VP
------------VERB - sat
------------PP
----------------ADP - on
----------------NP
--------------------DET - the
--------------------NOUN - couch
----PUNCT - .


Time flies like an arrow. -> 1 parse(s)
ROOT
----S
--------NP - Time
--------VP
------------VERB - flies
------------PP
----------------ADP - like
----------------NP
--------------------DET - an
--------------------NOUN - arrow
----PUNCT - .


The spy saw the cop with the telescope. -> 2 parse(s)
ROOT
----S
--------NP
------------DET - The
------------NOUN - spy
--------VP
------------VERB - saw
------------NP
----------------NP
--------------------DET - the
--------------------NOUN - cop
----------------PP
--------------------ADP - with
--------------------NP
------------------------DET - the
------------------------NOUN - telescope
----PUNCT - .


ROOT
----S
--------NP
------------DET - The
------------NOUN - 

In [27]:
!pip install benepar

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 23.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [28]:
!python -m spacy download en_core_web_sm

Defaulting to user installation because normal site-packages is not writeable
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 0.1/12.8 MB 3.3 MB/s eta 0:00:04
      --------------------------------------- 0.3/12.8 MB 3.8 MB/s eta 0:00:04
     - -------------------------------------- 0.5/12.8 MB 3.7 MB/s eta 0:00:04
     -- ------------------------------------- 0.6/12.8 MB 3.7 MB/s eta 0:00:04
     -- ------------------------------------- 0.8/12.8 MB 3.8 MB/s eta 0:00:04
     --- ------------------------------------ 1.0/12.8 MB 3.7 MB/s eta 0:00:04
     --- ------------------------------------ 1.3/12.8 MB 4.0 MB/s eta 0:00:03
     ---- ----------------------------------- 1.6/12.8 MB 4.4 MB/s eta 0:00:03
     ----- ---------------------------------- 1.8/12.8 MB 4.3 MB/s eta 0:00:03
     ----- ---------------------------------- 1.8/12.8 MB 4.3 MB/s eta 0:00:03
     ------ --------------------------------- 2.2/12.8 MB 4.


[notice] A new release of pip is available: 23.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [29]:
import os
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"

import benepar
import spacy

# Optional: ensure benepar model is downloaded
benepar.download('benepar_en3')

nlp = spacy.load("en_core_web_sm")
nlp.add_pipe('benepar', config={'model': 'benepar_en3'})

sentences = [
    "The cat sat on the couch.",
    "Time flies like an arrow.",
    "The spy saw the cop with the telescope.",
    "The spy saw the cop with the revolver.",
]

for text in sentences:
    doc = nlp(text)
    for sent in doc.sents:
        print(text)
        print(sent._.parse_string)
        print()

[nltk_data] Downloading package benepar_en3 to C:\Users\PC CABA
[nltk_data]     DZ\AppData\Roaming\nltk_data...
[nltk_data]   Package benepar_en3 is already up-to-date!
You're using a T5TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


The cat sat on the couch.
(S (NP (DT The) (NN cat)) (VP (VBD sat) (PP (IN on) (NP (DT the) (NN couch)))) (. .))

Time flies like an arrow.
(S (NP (NN Time)) (VP (VBZ flies) (PP (IN like) (NP (DT an) (NN arrow)))) (. .))

The spy saw the cop with the telescope.
(S (NP (DT The) (NN spy)) (VP (VBD saw) (NP (DT the) (NN cop)) (PP (IN with) (NP (DT the) (NN telescope)))) (. .))

The spy saw the cop with the revolver.
(S (NP (DT The) (NN spy)) (VP (VBD saw) (NP (DT the) (NN cop)) (PP (IN with) (NP (DT the) (NN revolver)))) (. .))



In [30]:
!pip install svglib reportlab

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 23.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [31]:
from nltk import Tree
import svgling

trees_str = [
    "(S (NP (DT The) (NN cat)) (VP (VBD sat) (PP (IN on) (NP (DT the) (NN couch)))) (. .))",
    "(S (NP (NN Time)) (VP (VBZ flies) (PP (IN like) (NP (DT an) (NN arrow)))) (. .))",
    "(S (NP (DT The) (NN spy)) (VP (VBD saw) (NP (DT the) (NN cop)) (PP (IN with) (NP (DT the) (NN telescope)))) (. .))",
    "(S (NP (DT The) (NN spy)) (VP (VBD saw) (NP (DT the) (NN cop)) (PP (IN with) (NP (DT the) (NN revolver)))) (. .))"
]

for idx, parse_str in enumerate(trees_str, 1):
    t = Tree.fromstring(parse_str)
    img = svgling.draw_tree(t)
    
    # Save as SVG file using _repr_svg_()
    with open(f"tree_{idx}.svg", "w", encoding="utf-8") as f:
        f.write(img._repr_svg_())

print("Saved all trees as SVG files!")

Saved all trees as SVG files!
